# Syndrome data discovery

This notebook investigates the supplied qec_syndromes Bronze archive without modifying it.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import ast
import hashlib
import json
import re

import pandas as pd
from IPython.display import display

archive_path = Path(
    "/course-data/raw/source=qec_syndromes/syndromes_dataset.zip"
)
manifest_path = Path("/course-data/metadata/bundle-manifest.json")

assert archive_path.is_file(), "Archive not found: {}".format(archive_path)
assert manifest_path.is_file(), "Manifest not found: {}".format(manifest_path)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

expected_object = next(
    item for item in manifest["objects"]
    if item["source"] == "qec_syndromes"
)

hasher = hashlib.sha256()

with archive_path.open("rb") as file:
    while True:
        chunk = file.read(1024 * 1024)

        if chunk == b"":
            break

        hasher.update(chunk)

actual_sha256 = hasher.hexdigest()
actual_bytes = archive_path.stat().st_size

# Create frame for clean output.
integrity = pd.Series({
    "expected_bytes": expected_object["bytes"],
    "actual_bytes": actual_bytes,
    "size_matches": actual_bytes == expected_object["bytes"],
    "expected_sha256": expected_object["sha256"],
    "actual_sha256": actual_sha256,
    "checksum_matches": actual_sha256 == expected_object["sha256"],
})
display(integrity.to_frame("value"))


## My findings: Bronze integrity

From results we infer:

1) Object exists.
2) SHA-256 match the supplied manifest.



In [ ]:
# Basic safety check of path from zip
def is_safe_member_path(name):
    normalized = name.replace("\\", "/")
    member_path = PurePosixPath(normalized)
    
    return (
        not member_path.is_absolute()
        and ".." not in member_path.parts
    )

with ZipFile(archive_path) as archive:
    member_names = archive.namelist()
    # Check if Zip is corrupted
    bad_crc_member = archive.testzip()
    
    member_inventory = pd.DataFrame([
        {
            "member": info.filename,
            "original_bytes": info.file_size,
            "compressed_bytes": info.compress_size,
            "safe_path": is_safe_member_path(info.filename),
        }
        for info in archive.infolist()
    ])

csv_members = sorted(
    name for name in member_names
    if name.lower().endswith(".csv")
)
unsafe_members = [
    name for name in member_names
    if not is_safe_member_path(name)
]

display(member_inventory)
display(pd.Series({
    "all_members": len(member_names),
    "csv_members": len(csv_members),
    "unsafe_members": len(unsafe_members),
    "duplicate_member_names": len(member_names) - len(set(member_names)),
    "first_bad_crc_member": (
      bad_crc_member
      if bad_crc_member is not None
      else "No CRC errors"
  )
}).to_frame("value"))

assert len(csv_members) == 7
assert not unsafe_members
assert len(member_names) == len(set(member_names))
assert bad_crc_member is None


## My findings: archive contents and path safety

_Write here how many CSV files exist, whether their paths are safe, and whether the ZIP integrity check passed._

All checks passed as expected:
1) 7 csv files
2) 0 duplicates
3) No corrupted files and unsafe members.

In [ ]:
filename_pattern = re.compile(
    r"^d-(?P<distance>\d+)_pfr-(?P<fault_rate>\d+(?:\.\d+)?)_nb-(?P<count>\d+)(?P<unit>[KMG]?)\.csv$",
    flags=re.IGNORECASE,
)
size_multipliers = {
    "M": 1_000_000,
}

def filename_metadata(member):
    match = filename_pattern.fullmatch(PurePosixPath(member).name)
    if match is None:
        return {
            "filename_valid": False,
            "distance": None,
            "physical_fault_rate": None,
            "nominal_observations": None,
        }

    fields = match.groupdict()
    return {
        "filename_valid": True,
        "distance": int(fields["distance"]),
        "physical_fault_rate": float(fields["fault_rate"]),
        "nominal_observations": (
            int(fields["count"])
            * size_multipliers[fields["unit"].upper()]
        ),
    }

frames = []
file_details = []

with ZipFile(archive_path) as archive:
    for member in csv_members:
        metadata = filename_metadata(member)
        with archive.open(member) as file:
            frame = pd.read_csv(file)

        source_columns = frame.columns.tolist()
        frame["source_member"] = member
        frame["source_row_number"] = range(1, len(frame) + 1)
        frame["distance"] = metadata["distance"]
        frame["physical_fault_rate"] = metadata["physical_fault_rate"]
        frame["nominal_observations"] = metadata["nominal_observations"]
        frames.append(frame)

        file_details.append({
            "source_member": member,
            "rows": len(frame),
            "source_columns": source_columns,
            **metadata,
        })

syndromes_raw = pd.concat(frames, ignore_index=True)
file_inventory = pd.DataFrame(file_details)

display(file_inventory)
display(syndromes_raw.head())
print(
    "Loaded {:,} aggregate rows from {} files."
    .format(len(syndromes_raw), len(frames))
)


Each filename encodes the following information:

  1. d: surface-code distance.
  2. pfr: physical fault rate.
  3. nb: nominal number of simulated observations.

  All seven CSV filenames follow the expected pattern.

  Each row in the syndrome dataset represents one aggregate observation containing:

  1. labels: the logical-error label, either 0 or 1.
  2. syndromes: a 4×4 sequence of binary syndrome values.
  3. quantity: the number of simulated shots represented by that row.

  A row does not represent only one shot but a number of shots with the same syndrome and logical-error label.

In [ ]:
expected_columns = ["labels", "syndromes", "quantity"]

schema_check = file_inventory[
    ["source_member", "source_columns"]
].copy()
schema_check["matches_expected_schema"] = schema_check[
    "source_columns"
].apply(lambda columns: columns == expected_columns)

column_summary = pd.DataFrame({
    "pandas_dtype": syndromes_raw[expected_columns].dtypes.astype(str),
    "missing_values": syndromes_raw[expected_columns].isna().sum(),
})

display(schema_check)
display(column_summary)
display(
    syndromes_raw["labels"]
    .value_counts(dropna=False)
    .rename("rows")
    .to_frame()
)


## My findings: schema, types, and missing values


All files use the header provided in the documentation. 
Data types according to pandas are:
1) labels: int64
2) syndromes: object
3) quantity int64


In [ ]:
def check_syndrome(value):
    if not isinstance(value, str):
        return False, "not a string", None

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError) as error:
        return False, "cannot parse: {}".format(error), None

    if not isinstance(parsed, (tuple, list)) or len(parsed) != 4:
        return False, "outer structure must contain four rounds", parsed

    if not all(
        isinstance(round_values, (tuple, list))
        and len(round_values) == 4
        for round_values in parsed
    ):
        return False, "each round must contain four values", parsed

    if not all(
        type(bit) is int and bit in (0, 1)
        for round_values in parsed
        for bit in round_values
    ):
        return False, "syndrome values must be binary integers", parsed

    return True, None, parsed

validation = syndromes_raw.copy()
inspection = validation["syndromes"].map(check_syndrome)
validation["syndrome_valid"] = inspection.map(
    lambda result: result[0]
)
validation["syndrome_issue"] = inspection.map(
    lambda result: result[1]
)
validation["parsed_syndrome"] = inspection.map(
    lambda result: result[2]
)

quantity = pd.to_numeric(validation["quantity"], errors="coerce")
validation["label_valid"] = validation["labels"].isin([0, 1])
validation["quantity_numeric"] = quantity.notna()
validation["quantity_integer"] = (
    quantity.notna() & quantity.mod(1).eq(0)
)
validation["quantity_positive"] = quantity.gt(0)
validation["record_valid"] = validation[[
    "syndrome_valid",
    "label_valid",
    "quantity_numeric",
    "quantity_integer",
    "quantity_positive",
]].all(axis=1)

validation_summary = pd.Series({
    "total_rows": len(validation),
    "invalid_syndrome_rows": (~validation["syndrome_valid"]).sum(),
    "invalid_label_rows": (~validation["label_valid"]).sum(),
    "non_numeric_quantity_rows": (~validation["quantity_numeric"]).sum(),
    "non_integer_quantity_rows": (~validation["quantity_integer"]).sum(),
    "non_positive_quantity_rows": (~validation["quantity_positive"]).sum(),
    "invalid_records_total": (~validation["record_valid"]).sum(),
})

display(validation_summary.to_frame("count"))
display(validation.loc[
    ~validation["record_valid"],
    [
        "source_member",
        "source_row_number",
        "labels",
        "syndromes",
        "quantity",
        "syndrome_issue",
    ],
].head(20))


## My findings: record validation

Every syndrome follows a 4-by-4 binary structure as the documentation indicates.
All labels are binary as expected, all quantities are positive integers.

There are no invalid tuples in the dataset.


In [ ]:
per_file_counts = (
    validation
    .groupby("source_member", as_index=False)
    .agg(
        aggregate_rows=("quantity", "size"),
        weighted_observations=("quantity", "sum"),
        nominal_observations=("nominal_observations", "first"),
        distance=("distance", "first"),
        physical_fault_rate=("physical_fault_rate", "first"),
    )
)
per_file_counts["weighted_total_matches_filename"] = (
    per_file_counts["weighted_observations"]
    == per_file_counts["nominal_observations"]
)

overall_counts = pd.Series({
    "source_files": len(per_file_counts),
    "aggregate_rows": len(validation),
    "weighted_observations": validation["quantity"].sum(),
    "expected_source_files": 7,
    "expected_aggregate_rows": 75_598,
    "expected_weighted_observations": 70_000_000,
})

display(per_file_counts)
display(overall_counts.to_frame("value"))


## My findings: count reconciliation

Weightes observations and aggregate rows are 70 million and  75.598 in number as expected.
The overall total match the brief that was given.

In [ ]:
source_locator_columns = [
    "source_member",
    "source_row_number",
]
duplicate_locators = validation.duplicated(
    source_locator_columns,
    keep=False,
)

syndrome_label_counts = (
    validation
    .groupby(
        ["distance", "physical_fault_rate", "syndromes"],
        dropna=False,
    )
    .agg(
        distinct_labels=("labels", "nunique"),
        rows=("labels", "size"),
        weighted_observations=("quantity", "sum"),
    )
    .reset_index()
)
same_syndrome_different_label = syndrome_label_counts.loc[
    syndrome_label_counts["distinct_labels"] > 1
]

bronze_object_key = (
    "bronze/source=qec_syndromes/syndromes_dataset.zip"
)

def candidate_source_record_id(member, row_number):
    locator = "|".join([
        actual_sha256,
        bronze_object_key,
        member,
        str(row_number),
    ])
    digest = hashlib.sha256(locator.encode("utf-8")).hexdigest()
    return "qec-syn-" + digest

validation["candidate_source_record_id"] = [
    candidate_source_record_id(member, row_number)
    for member, row_number in zip(
        validation["source_member"],
        validation["source_row_number"],
    )
]

identifier_summary = pd.Series({
    "duplicate_source_locators": duplicate_locators.sum(),
    "distinct_syndrome_strings": validation[
        "syndromes"
    ].nunique(dropna=False),
    "same_syndrome_different_label_groups": len(
        same_syndrome_different_label
    ),
    "candidate_ids_unique": validation[
        "candidate_source_record_id"
    ].is_unique,
    "candidate_ids_non_null": validation[
        "candidate_source_record_id"
    ].notna().all(),
})

display(identifier_summary.to_frame("value"))
display(same_syndrome_different_label.head(20))
display(validation[[
    "candidate_source_record_id",
    "source_member",
    "source_row_number",
    "labels",
    "quantity",
]].head())


## My findings: identifiers and label ambiguity

There are 31,941 distinct syndrome strings among 75,598 aggregate rows. Therefore, the syndrome value alone is not a unique identifier.

There are 25,159 groups in which the same syndrome and physical fault rate occur with both logical-error labels. These cases are valid because the same syndrome can produce different logical outcomes.

The combination of the archive member and source row number uniquely locates every original record. The candidate source_record_id is also unique and non-null. It is stable because it is calculated from the Bronze archive hash, Bronze object name, archive member, and source row number.


In [ ]:
discovery_summary = pd.Series({
    "bronze_checksum_valid": (
        actual_sha256 == expected_object["sha256"]
    ),
    "safe_archive_paths": not unsafe_members,
    "csv_files": len(csv_members),
    "all_schemas_match": schema_check[
        "matches_expected_schema"
    ].all(),
    "aggregate_rows": len(validation),
    "weighted_observations": validation["quantity"].sum(),
    "valid_records": validation["record_valid"].sum(),
    "invalid_records": (~validation["record_valid"]).sum(),
    "candidate_ids_unique": validation[
        "candidate_source_record_id"
    ].is_unique,
    "same_syndrome_different_label_groups": len(
        same_syndrome_different_label
    ),
})
display(discovery_summary.to_frame("value"))


## My final discovery conclusions

The Bronze archive has the expected size and SHA-256 checksum. It contains seven CSV files with safe member paths, unique names, and valid CRC checks.

Each filename records the surface-code distance, physical fault rate, and nominal number of simulated observations. All seven filenames follow the expected pattern.

One CSV row represents an aggregate syndrome observation. It contains one logical-error label, one 4-by-4 binary syndrome, and a quantity showing how many simulated shots the row represents.

All 75,598 rows have the expected schema and pass the syndrome, label, and quantity checks. The quantities represent 70,000,000 simulated observations in total, and every file matches the nominal count in its filename. No syndrome records need to be rejected during this stage.

Syndrome alone cannot be used as a key because syndrome values repeat and can occur with both labels. A stable source_record_id should instead use the Bronze archive hash, object name, archive member, and source row number.

For the Silver table, the 4-by-4 syndrome will be flattened into 16 bytes in round-first order. The filename supplies the physical fault rate and experiment identifier. The label becomes a Boolean value, while quantity remains an integer weight and must not be expanded into separate rows.

